In [1]:
# This is a producer notebook - will send messages to Kafka / Red Panda

import pandas as pd

In [2]:
url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-11.parquet'

In [3]:
columns = ['PULocationID', 'DOLocationID', 'trip_distance', 'total_amount', 'tpep_pickup_datetime']
df = pd.read_parquet(url, columns=columns).head(1000)

In [4]:
df.head()

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50


In [6]:
from models import Ride, ride_from_row, ride_serializer

In [7]:
# read 1st row from our parquet/dataframe and check whats inside:

ride = ride_from_row(df.iloc[1])
ride

# tpep_pickup_datetime=1761958147000 because timestamp in milliseconds

Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000)

In [8]:
from kafka import KafkaProducer

server = 'localhost:9092'

producer = KafkaProducer(
    bootstrap_servers=[server],
    value_serializer=ride_serializer
)
producer
# <kafka.producer.kafka.KafkaProducer at 0x764f7248ca10>

In [9]:
topic_name = 'rides'

producer.send(topic_name, value=ride)
producer.flush()
# flush to run, otherwise not executed and kafka topic will be empty

In [9]:
# let me send 20 rides first:
df_small = df.iloc[:20]
df_small

,PULocationID,DOLocationID,trip_distance,total_amount,tpep_pickup_datetime
0,43,186,1.68,22.15,2025-11-01 00:13:25
1,142,237,2.28,24.94,2025-11-01 00:49:07
2,163,238,2.70,25.62,2025-11-01 00:07:19
3,138,261,12.87,86.14,2025-11-01 00:00:00
4,138,37,8.40,48.65,2025-11-01 00:18:50
5,90,100,0.85,16.45,2025-11-01 00:21:11
6,142,170,3.01,25.85,2025-11-01 00:07:31
7,237,144,3.82,57.54,2025-11-01 00:46:52
8,162,161,0.89,12.95,2025-11-01 00:56:59
9,234,162,2.28,38.68,2025-11-01 00:10:43


In [10]:
import time

t0 = time.time()

for _, row in df_small.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

Sent: Ride(PULocationID=43, DOLocationID=186, trip_distance=1.68, total_amount=22.15, tpep_pickup_datetime=1761956005000)
Sent: Ride(PULocationID=142, DOLocationID=237, trip_distance=2.28, total_amount=24.94, tpep_pickup_datetime=1761958147000)
Sent: Ride(PULocationID=163, DOLocationID=238, trip_distance=2.7, total_amount=25.62, tpep_pickup_datetime=1761955639000)
Sent: Ride(PULocationID=138, DOLocationID=261, trip_distance=12.87, total_amount=86.14, tpep_pickup_datetime=1761955200000)
Sent: Ride(PULocationID=138, DOLocationID=37, trip_distance=8.4, total_amount=48.65, tpep_pickup_datetime=1761956330000)
Sent: Ride(PULocationID=90, DOLocationID=100, trip_distance=0.85, total_amount=16.45, tpep_pickup_datetime=1761956471000)
Sent: Ride(PULocationID=142, DOLocationID=170, trip_distance=3.01, total_amount=25.85, tpep_pickup_datetime=1761955651000)
Sent: Ride(PULocationID=237, DOLocationID=144, trip_distance=3.82, total_amount=57.54, tpep_pickup_datetime=1761958012000)
Sent: Ride(PULocatio

In [11]:
# now send all data - a 1000 rows in our case :
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 5 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   PULocationID          1000 non-null   int32         
 1   DOLocationID          1000 non-null   int32         
 2   trip_distance         1000 non-null   float64       
 3   total_amount          1000 non-null   float64       
 4   tpep_pickup_datetime  1000 non-null   datetime64[us]
dtypes: datetime64[us](1), float64(2), int32(2)
memory usage: 31.4 KB


In [11]:
import time

t0 = time.time()

for _, row in df.iterrows():
    ride = ride_from_row(row)
    producer.send(topic_name, value=ride)
    # print(f"Sent: {ride}")
    time.sleep(0.01)

producer.flush()

t1 = time.time()
print(f'took {(t1 - t0):.2f} seconds')

took 10.49 seconds


In [12]:
# the end